# Post-Stage B Experiments — Google Colab GPU Runner

Thin orchestration notebook calling reusable modules in `src/bangla_smish/`.

### Execution Policy
- **Stage B synthetic generation is permanently stopped.**
- **Default mode is `RUN_MODE = "gpu_smoke_test"`** so running the notebook first validates CUDA, dependencies, Drive mounting, manifest verification, checkpointing, and safe resume on a tiny subset before consuming significant compute units.
- Full Experiments A, B, C, and D require owner authorization, a fresh artifact run ID, and `AUTHORIZE_FULL_TRAINING = True`. Keep a run ID fixed when resuming that same run.
  - **Experiment A**: Unbalanced Real Baseline (`eligible_train`, `N = 3,999`)
  - **Experiment B**: Class-Weighted Real Baseline (`eligible_train`, `N = 3,999`, fold-specific training loss weights)
  - **Experiment C**: Clean Stage B Promotional Augmentation (`eligible_train + Batches 1–7`, `N = 4,344`, synthetic train-only)
  - **Experiment D**: Full Stage B Balanced Pool (`eligible_train + Pilot 300 + Batches 1–7`, `N = 4,644`, synthetic train-only)
- **Security**: Never place tokens, service account keys, or credentials in this notebook or Git. Google Drive is mounted interactively via `google.colab.drive.mount` only after your authorization.

In [ ]:
#@title 1. Runtime Configuration & Safety Mode
RUN_MODE = 'gpu_smoke_test'  #@param ['gpu_smoke_test', 'experiments_abcd']
AUTHORIZE_FULL_TRAINING = False  #@param {type:'boolean'}
REQUIRE_GPU = True  #@param {type:'boolean'}
USE_GOOGLE_DRIVE = True  #@param {type:'boolean'}
DRIVE_WORKSPACE_REL = 'bangla_smish_post_stage_b_v2'  #@param {type:'string'}
ARTIFACT_RUN_ID = 'audit-v2-smoke-01'  #@param {type:'string'}
RESUME = True  #@param {type:'boolean'}
RUN_INTERRUPTION_RESUME_CHECK = True  #@param {type:'boolean'}

def validate_launch_settings():
    import re
    if RUN_MODE not in {'gpu_smoke_test', 'experiments_abcd'}:
        raise ValueError(f'Unknown run mode {RUN_MODE!r}')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', ARTIFACT_RUN_ID):
        raise ValueError('Set ARTIFACT_RUN_ID to a unique directory name using letters, digits, hyphens, or underscores.')
    if RUN_MODE == 'experiments_abcd' and not AUTHORIZE_FULL_TRAINING:
        raise RuntimeError('Full training requires explicit project owner authorization.')

validate_launch_settings()
print(f'Configured {RUN_MODE=} {ARTIFACT_RUN_ID=} {USE_GOOGLE_DRIVE=} {RESUME=}')

In [ ]:
#@title 2. Verify Runtime Environment & GPU Availability
import os, platform, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print('Python:', sys.version.split()[0], '| Platform:', platform.platform(), '| IN_COLAB:', IN_COLAB)

if shutil.which('nvidia-smi'):
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip())
else:
    print('nvidia-smi not found in PATH.')

import torch
cuda_ok = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| CUDA available:', cuda_ok)
if cuda_ok:
    props = torch.cuda.get_device_properties(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| VRAM: {props.total_memory / (1024**3):.2f} GB')
elif REQUIRE_GPU:
    raise RuntimeError('CUDA GPU is required. In Colab: Runtime -> Change runtime type -> T4 / L4 / A100 GPU.')

disk = shutil.disk_usage('/content' if Path('/content').exists() else '.')
print(f'Free local disk: {disk.free / (1024**3):.2f} GB')

In [ ]:
#@title 3. Authorize Google Drive Mount & Locate Project Workspace
from pathlib import Path
import sys

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    # Explicit interactive OAuth authorization by the user; no credentials stored in code.
    drive.mount('/content/drive', force_remount=False)
    WORKSPACE_ROOT = Path('/content/drive/MyDrive') / DRIVE_WORKSPACE_REL
else:
    WORKSPACE_ROOT = Path('/content/bangla_smish_post_stage_b_v2')

SRC_DIR = WORKSPACE_ROOT / 'src'
BUNDLE_DIR = WORKSPACE_ROOT / 'data' / 'processed' / 'post_stage_b_bundle_v2'
CONFIG_DIR = WORKSPACE_ROOT / 'experiments' / 'colab' / 'configs'
ARTIFACTS_DIR = WORKSPACE_ROOT / 'artifacts' / RUN_MODE / ARTIFACT_RUN_ID
cfg_filename = 'gpu_smoke_test_config.json' if RUN_MODE == 'gpu_smoke_test' else 'post_stage_b_abcd_config.json'
CONFIG_PATH = CONFIG_DIR / cfg_filename

print('WORKSPACE_ROOT:', WORKSPACE_ROOT)
print('BUNDLE_DIR exists:', BUNDLE_DIR.exists(), '| ARTIFACTS_DIR:', ARTIFACTS_DIR)
if not SRC_DIR.is_dir() or not BUNDLE_DIR.is_dir() or not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f'Missing source, versioned dataset bundle, or configuration under {WORKSPACE_ROOT}. '
        'Prepare the v2 Drive kit locally before staging it in a fresh workspace.'
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
#@title 4. Install Compatible Dependencies (Preserving Colab CUDA PyTorch)
import json, subprocess, sys, tempfile
from importlib.metadata import PackageNotFoundError, distribution, version
from packaging.requirements import Requirement
from packaging.version import Version
from runpy import run_path
# Read the dependency declaration before importing the package and its dependencies.
model_setup = run_path(str(SRC_DIR / 'bangla_smish' / 'post_stage_b_tokenization.py'))

REQUIRED_PACKAGES = [
    'transformers>=4.57.1,<5', 'datasets>=2.20,<5', 'accelerate>=0.33,<2',
    'safetensors>=0.4', 'sentencepiece>=0.2,<1',
    'numpy>=1.26', 'pandas>=2.1', 'scipy>=1.11', 'scikit-learn>=1.4', 'pyarrow>=15.0',
]
if model_setup['BANGLABERT_MODEL'] in json.loads(CONFIG_PATH.read_text(encoding='utf-8'))['models']:
    REQUIRED_PACKAGES.append(model_setup['BANGLABERT_NORMALIZER_REQUIREMENT'])

def requirement_is_satisfied(spec):
    req = Requirement(spec)
    try:
        dist = distribution(req.name)
    except PackageNotFoundError:
        return False
    if req.url:
        origin = json.loads(dist.read_text('direct_url.json') or '{}')
        return (origin.get('url') == 'https://github.com/csebuetnlp/normalizer'
                and origin.get('vcs_info', {}).get('commit_id') == model_setup['BANGLABERT_NORMALIZER_COMMIT'])
    return Version(dist.version) in req.specifier

torch_version = version('torch')
if Version(torch_version) < Version('2.6'):
    raise RuntimeError('Select a Colab runtime with PyTorch 2.6+ for Transformers checkpoint loading.')
to_install = [spec for spec in REQUIRED_PACKAGES if not requirement_is_satisfied(spec)]
if to_install:
    with tempfile.TemporaryDirectory() as temp_dir:
        constraint = Path(temp_dir) / 'constraints.txt'
        constraint.write_text(f'torch=={torch_version}\n', encoding='utf-8')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), *REQUIRED_PACKAGES])
    raise RuntimeError('Dependencies installed. Restart the Colab session in the same runtime, then rerun the notebook.')
for spec in REQUIRED_PACKAGES:
    name = Requirement(spec).name
    print(f'{name}: {version(name)}')

In [ ]:
#@title 5. Load & Verify Dataset Manifest + Experiment Config
import json
from bangla_smish.post_stage_b_data import sha256_file, verify_manifest
from bangla_smish.post_stage_b_runner import collect_environment_info, load_config, seed_everything

manifest = verify_manifest(BUNDLE_DIR, verify_checksums=True)
print('Verified manifest_version:', manifest['manifest_version'])
print('Manifest SHA256:', sha256_file(BUNDLE_DIR / 'dataset_manifest.json'))
print('Integrity checks:', json.dumps(manifest['integrity_checks'], indent=2))
print('Cohort counts:', json.dumps({k: v.get('by_class', v) for k, v in manifest['cohort_counts'].items()}, indent=2))

cfg = load_config(CONFIG_PATH)
if cfg.run_mode != RUN_MODE:
    raise ValueError('Notebook run mode and loaded configuration disagree.')
seed_everything(cfg.seed)
env_info = collect_environment_info()
print('Loaded config:', cfg.run_name, '| config_hash:', cfg.config_hash())
print('Environment snapshot:', json.dumps(env_info, indent=2))

In [ ]:
#@title 6. Execute Pipeline (GPU Smoke Test OR Experiments A, B, C, D with Safe Resume)
import json
from bangla_smish.post_stage_b_runner import run_post_stage_b_pipeline

validate_launch_settings()
if cfg.run_mode != RUN_MODE or ARTIFACTS_DIR != WORKSPACE_ROOT / 'artifacts' / RUN_MODE / ARTIFACT_RUN_ID:
    raise ValueError('Rerun setup cells after changing launch settings.')
if RUN_MODE == 'gpu_smoke_test' and RESUME and RUN_INTERRUPTION_RESUME_CHECK and not (ARTIFACTS_DIR / 'run_state.json').exists():
    print('--- Step 6a: Running 1 unit and simulating runtime interruption ---')
    partial = run_post_stage_b_pipeline(
        bundle_dir=BUNDLE_DIR,
        artifacts_dir=ARTIFACTS_DIR,
        cfg=cfg,
        resume=RESUME,
        verify_checksums=True,
        stop_after_units=1,
    )
    print('Completed after partial pass:', partial['completed_unit_count'])
    print('--- Step 6b: Resuming from persistent Drive checkpoint/state ---')

summary = run_post_stage_b_pipeline(
    bundle_dir=BUNDLE_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    cfg=cfg,
    resume=RESUME,
    verify_checksums=True,
)
print('Completed units total:', summary['completed_unit_count'])

In [ ]:
#@title 7. Inspect Exported Results, Logs & Persistent Storage Artifacts
import pandas as pd
from pathlib import Path

exports_dir = ARTIFACTS_DIR / 'exports'
logs_path = ARTIFACTS_DIR / 'logs' / 'runner.log'
print('Exported files:', [p.name for p in sorted(exports_dir.glob('*'))])
print('Run state:', (ARTIFACTS_DIR / 'run_state.json').read_text(encoding='utf-8'))

csv_path = exports_dir / 'aggregated_results.csv'
if csv_path.exists() and csv_path.stat().st_size > 0:
    display_df = pd.read_csv(csv_path)
    print(display_df.to_string(index=False))

print('\nLast 15 log lines:')
if logs_path.exists():
    print('\n'.join(logs_path.read_text(encoding='utf-8').splitlines()[-15:]))